# Vision Transformer B/16 Training - v2

**Model:** Vision Transformer Base with 16×16 patches (ViT-B/16)  
**Architecture:** Pure transformer with self-attention mechanism  
**Dataset:** Kermany OCT2017 (verified clean)  
**Validation:** 15% stratified split (11,521 images)

## ViT-B/16 Architecture

Vision Transformer processes images as sequences of patches:
1. **Patch Embedding:** 224×224 image → 196 patches (16×16 each)
2. **Transformer Encoder:** 12 layers with multi-head self-attention
3. **Classification Head:** Global representation → class prediction

## ViT-Specific Training

Transformers require different hyperparameters than CNNs:
- **Optimizer:** AdamW (better for transformers)
- **Scheduler:** Cosine annealing (smooth decay)
- **Learning Rate:** Lower than CNNs (3e-4 vs 1e-3)
- **Weight Decay:** Higher than CNNs (0.05 vs 1e-4)
- **Mixed Precision:** AMP for faster training

In [1]:
# IMPORTS
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torch.cuda.amp import autocast, GradScaler
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import vit_b_16, ViT_B_16_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
from pathlib import Path
import numpy as np
import time
from tqdm import tqdm
from collections import Counter
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Imports successful")

Imports successful


In [2]:
# HELPER FUNCTIONS

def get_next_serial_number(checkpoint_dir):
    """Automatically detect the next available serial number for checkpoints."""
    import re
    from pathlib import Path
    
    checkpoint_dir = Path(checkpoint_dir)
    if not checkpoint_dir.exists():
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        return 1
    
    existing = list(checkpoint_dir.glob("*.pth"))
    if not existing:
        return 1
    
    serial_numbers = []
    for f in existing:
        match = re.match(r'^(\d+)_', f.name)
        if match:
            serial_numbers.append(int(match.group(1)))
    
    return max(serial_numbers) + 1 if serial_numbers else 1


def save_checkpoint(model, optimizer, scheduler, scaler, epoch, metrics, is_best,
                   checkpoint_dir, serial_number, model_name, seed, mode='intermediate'):
    """Save model checkpoint with comprehensive training state and metrics."""
    from datetime import datetime
    from pathlib import Path
    
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    serial_str = f"{serial_number:02d}"
    
    filename = f"{serial_str}_{model_name}_seed{seed}_epoch{epoch}_{mode}_{timestamp}.pth"
    filepath = checkpoint_dir / filename
    
    checkpoint = {
        'serial_number': serial_number,
        'model_name': model_name,
        'seed': seed,
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict() if scaler else None,
        'metrics': metrics,
        'is_best': is_best,
        'mode': mode,
        'timestamp': timestamp
    }
    
    torch.save(checkpoint, filepath)
    print(f"💾 Saved {mode}: {filename}")
    return filepath


def create_stratified_split(dataset, val_ratio=0.15, seed=42):
    """
    Create stratified train/validation split maintaining class balance.
    Uses pre-loaded labels from ImageFolder.targets for efficiency.
    """
    labels = np.array(dataset.targets)
    indices = np.arange(len(labels))
    
    train_idx, val_idx = train_test_split(
        indices,
        test_size=val_ratio,
        stratify=labels,
        random_state=seed
    )
    
    return train_idx, val_idx


def is_better_model(new_score, new_loss, new_acc, new_epoch,
                    best_score, best_loss, best_acc, best_epoch,
                    eps=1e-9):
    """
    Deterministic model comparison with clear priority hierarchy.
    
    Priority order:
    1. Composite score (primary metric)
    2. Validation loss (tie-breaker)
    3. Validation accuracy (secondary tie-breaker)
    4. Epoch number (prefer later epochs for stability)
    
    Returns True if new model outperforms current best.
    """
    if new_score > best_score + eps:
        return True
    
    if abs(new_score - best_score) <= eps:
        if new_loss < best_loss - eps:
            return True
        
        if abs(new_loss - best_loss) <= eps:
            if new_acc > best_acc + eps:
                return True
            
            if abs(new_acc - best_acc) <= eps:
                if new_epoch > best_epoch:
                    return True
    
    return False


def check_overfitting(train_acc, val_acc, train_loss, val_loss,
                     threshold_acc=10.0, threshold_loss=0.5):
    """Detect overfitting based on train-validation performance gaps."""
    acc_gap = train_acc - val_acc
    loss_gap = val_loss - train_loss
    
    is_overfitting = (acc_gap > threshold_acc) or (loss_gap > threshold_loss)
    
    return {
        'is_overfitting': is_overfitting,
        'acc_gap': acc_gap,
        'loss_gap': loss_gap,
        'severity': 'HIGH' if (acc_gap > 15.0 or loss_gap > 1.0) else 'MODERATE' if is_overfitting else 'NONE'
    }


print("Helper functions loaded")

Helper functions loaded


In [3]:
# CONFIGURATION

ROOT = Path(r"C:\Users\Ajant\Documents\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training")

CHECKPOINT_DIR = ROOT / "Checkpoints"
DATASET_ROOT = ROOT / "Data_Kermany_OCT2017"
TRAIN_PATH = DATASET_ROOT / "train"
TEST_PATH = DATASET_ROOT / "test"

MODEL_NAME = "vit_b16"
NUM_EPOCHS = 50
SEED = 42

# ViT-specific hyperparameters (different from CNNs)
BATCH_SIZE = 64  # Larger batch for ViT
GRADIENT_ACCUMULATION_STEPS = 2  # Effective batch = 128
LEARNING_RATE = 0.0003  # Lower than CNNs (3e-4 vs 1e-3)
WEIGHT_DECAY = 0.05  # Higher than CNNs (0.05 vs 1e-4)
IMAGE_SIZE = 224
NUM_CLASSES = 4
CLASS_NAMES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']

VAL_SPLIT_RATIO = 0.15
SAVE_EVERY_N_EPOCHS = 5
OVERFITTING_CHECK_INTERVAL = 5

# Mixed precision training
USE_AMP = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Set seeds for reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SERIAL_NUMBER = get_next_serial_number(CHECKPOINT_DIR)

print("="*80)
print("CONFIGURATION - VISION TRANSFORMER B/16")
print("="*80)
print(f"Model: {MODEL_NAME}")
print(f"Serial: {SERIAL_NUMBER:02d} | Seed: {SEED} | Epochs: {NUM_EPOCHS}")
print(f"Device: {DEVICE}")
print(f"\nViT-Specific Settings:")
print(f"  Batch size: {BATCH_SIZE} × {GRADIENT_ACCUMULATION_STEPS} = {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS} effective")
print(f"  Learning rate: {LEARNING_RATE} (lower than CNNs)")
print(f"  Weight decay: {WEIGHT_DECAY} (higher than CNNs)")
print(f"  Mixed precision: {USE_AMP}")
print(f"\nValidation: {VAL_SPLIT_RATIO*100:.0f}% stratified split")
print("="*80)

CONFIGURATION - VISION TRANSFORMER B/16
Model: vit_b16
Serial: 12 | Seed: 42 | Epochs: 50
Device: cuda

ViT-Specific Settings:
  Batch size: 64 × 2 = 128 effective
  Learning rate: 0.0003 (lower than CNNs)
  Weight decay: 0.05 (higher than CNNs)
  Mixed precision: True

Validation: 15% stratified split


In [4]:
# DATASET VERIFICATION

print("="*80)
print("VERIFYING DATASET INTEGRITY")
print("="*80)

def list_files(root):
    """Get set of all image filenames in directory."""
    return set([p.name for p in Path(root).rglob("*.jpeg")])

train_files = list_files(TRAIN_PATH)
test_files = list_files(TEST_PATH)

print(f"Train files: {len(train_files):,}")
print(f"Test files: {len(test_files):,}")

overlap = train_files.intersection(test_files)
print(f"Overlap check: {len(overlap)} files")

if len(overlap) > 0:
    print("❌ WARNING: Train/test overlap detected!")
    print("Examples:", list(overlap)[:10])
    raise ValueError("Dataset contains train/test overlap")
else:
    print("✅ No overlap - dataset is clean")

print("="*80)

VERIFYING DATASET INTEGRITY
Train files: 55,792
Test files: 968
Overlap check: 0 files
✅ No overlap - dataset is clean


In [5]:
# DATASET LOADING

print("\n" + "="*80)
print("CREATING STRATIFIED TRAIN/VAL SPLIT")
print("="*80)

# Data transforms (ViT can handle stronger augmentation than CNNs)
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset for stratification
full_dataset = ImageFolder(root=str(TRAIN_PATH))
print(f"Total training images: {len(full_dataset):,}")

# Create stratified split
train_idx, val_idx = create_stratified_split(full_dataset, VAL_SPLIT_RATIO, SEED)

print(f"\nSplit created:")
print(f"  Training: {len(train_idx):,} images ({(1-VAL_SPLIT_RATIO)*100:.1f}%)")
print(f"  Validation: {len(val_idx):,} images ({VAL_SPLIT_RATIO*100:.1f}%)")

# Verify class balance
train_labels = [full_dataset.targets[i] for i in train_idx]
val_labels = [full_dataset.targets[i] for i in val_idx]

train_counts = Counter(train_labels)
val_counts = Counter(val_labels)

print("\nClass distribution:")
print(f"{'Class':<12} {'Training':>10} {'Validation':>12} {'Val %':>8}")
print("-" * 50)
for i, class_name in enumerate(CLASS_NAMES):
    train_count = train_counts[i]
    val_count = val_counts[i]
    val_pct = (val_count / (train_count + val_count)) * 100
    print(f"{class_name:<12} {train_count:>10,} {val_count:>12,} {val_pct:>7.1f}%")

# Create datasets with transforms
train_dataset_full = ImageFolder(root=str(TRAIN_PATH), transform=train_transform)
val_dataset_full = ImageFolder(root=str(TRAIN_PATH), transform=val_test_transform)

train_dataset = Subset(train_dataset_full, train_idx)
val_dataset = Subset(val_dataset_full, val_idx)

# Create dataloaders (adjusted for gradient accumulation)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"\nDataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print("="*80)


CREATING STRATIFIED TRAIN/VAL SPLIT
Total training images: 55,792

Split created:
  Training: 47,423 images (85.0%)
  Validation: 8,369 images (15.0%)

Class distribution:
Class          Training   Validation    Val %
--------------------------------------------------
CNV              19,006        3,354    15.0%
DME               5,862        1,034    15.0%
DRUSEN            3,280          579    15.0%
NORMAL           19,275        3,402    15.0%

DataLoaders created:
  Train batches: 741
  Val batches: 131


In [6]:
# MODEL INITIALIZATION

# Create ViT-B/16 model with pretrained ImageNet weights
model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)

# Replace classification head for OCT task
model.heads.head = nn.Linear(model.heads.head.in_features, NUM_CLASSES)
model = model.to(DEVICE)

# Class-balanced loss
class_weights = torch.tensor([
    len(train_labels) / (NUM_CLASSES * train_counts[i])
    for i in range(NUM_CLASSES)
], dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer: AdamW for transformers (better than Adam)
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Scheduler: Cosine annealing for transformers
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS
)

# Mixed precision scaler
scaler = GradScaler() if USE_AMP else None

print("="*80)
print("MODEL INITIALIZED")
print("="*80)
print(f"Architecture: Vision Transformer B/16")
print(f"Parameters: ~{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"Patch size: 16×16 (196 patches from 224×224 image)")
print(f"Transformer layers: 12")
print(f"Attention heads: 12")
print(f"\nOptimizer: AdamW")
print(f"Scheduler: CosineAnnealingLR")
print(f"Mixed precision (AMP): {USE_AMP}")
print(f"Class weights: {class_weights.cpu().numpy()}")
print("="*80)

MODEL INITIALIZED
Architecture: Vision Transformer B/16
Parameters: ~85.8M
Patch size: 16×16 (196 patches from 224×224 image)
Transformer layers: 12
Attention heads: 12

Optimizer: AdamW
Scheduler: CosineAnnealingLR
Mixed precision (AMP): True
Class weights: [0.62378985 2.0224752  3.614558   0.6150843 ]


In [7]:
# TRAINING LOOP WITH GRADIENT ACCUMULATION AND MIXED PRECISION

print("\n" + "="*80)
print(f"STARTING TRAINING - {MODEL_NAME.upper()}")
print("="*80)
print(f"Serial: {SERIAL_NUMBER:02d} | Seed: {SEED} | Epochs: {NUM_EPOCHS}")
print(f"Device: {DEVICE}")
print(f"Val size: {len(val_dataset):,} images")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print("="*80)

# Training history
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'val_f1': [], 'val_precision': [], 'val_recall': [],
    'composite_score': [],
    'learning_rates': [],
    'overfitting_checks': []
}

# Initialize best model tracking
best_composite_score = float('-inf')
best_val_acc = 0.0
best_val_loss = float('inf')
best_epoch = -1

start_time = time.time()

try:
    for epoch in range(NUM_EPOCHS):
        epoch_start = time.time()
        
        print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
        print("-" * 70)
        
        # TRAINING PHASE WITH GRADIENT ACCUMULATION
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        optimizer.zero_grad()
        
        for batch_idx, (images, labels) in enumerate(tqdm(train_loader, desc="Training", leave=False)):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            # Mixed precision forward pass
            if USE_AMP:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels) / GRADIENT_ACCUMULATION_STEPS
                
                scaler.scale(loss).backward()
                
                # Update weights every GRADIENT_ACCUMULATION_STEPS
                if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
            else:
                outputs = model(images)
                loss = criterion(outputs, labels) / GRADIENT_ACCUMULATION_STEPS
                loss.backward()
                
                if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    optimizer.step()
                    optimizer.zero_grad()
            
            train_loss += loss.item() * images.size(0) * GRADIENT_ACCUMULATION_STEPS
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
        
        train_loss = train_loss / len(train_dataset)
        train_acc = 100.0 * train_correct / train_total
        
        # VALIDATION PHASE
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validation", leave=False):
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                
                if USE_AMP:
                    with autocast():
                        outputs = model(images)
                        loss = criterion(outputs, labels)
                else:
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        val_loss = val_loss / len(val_dataset)
        val_acc = 100.0 * np.mean(np.array(all_preds) == np.array(all_labels))
        
        # Validate accuracy scale
        assert 0 <= train_acc <= 100, f"Train acc {train_acc:.2f} out of range"
        assert 0 <= val_acc <= 100, f"Val acc {val_acc:.2f} out of range"
        
        # Compute additional metrics
        val_f1 = f1_score(all_labels, all_preds, average='macro') * 100
        val_precision = precision_score(all_labels, all_preds, average='macro', zero_division=0) * 100
        val_recall = recall_score(all_labels, all_preds, average='macro', zero_division=0) * 100
        
        # Composite score for model selection
        composite_score = (
            0.40 * val_acc +
            0.25 * val_f1 +
            0.20 * (100 - min(val_loss * 10, 100)) +
            0.15 * max(0, 100 - abs(train_acc - val_acc) * 2)
        )
        
        # Update history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        history['val_precision'].append(val_precision)
        history['val_recall'].append(val_recall)
        history['composite_score'].append(composite_score)
        history['learning_rates'].append(optimizer.param_groups[0]['lr'])
        
        scheduler.step()
        
        # Best model selection
        is_best = is_better_model(
            new_score=composite_score,
            new_loss=val_loss,
            new_acc=val_acc,
            new_epoch=epoch + 1,
            best_score=best_composite_score,
            best_loss=best_val_loss,
            best_acc=best_val_acc,
            best_epoch=best_epoch
        )
        
        if is_best:
            best_composite_score = composite_score
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch + 1
            
            metrics = {
                'train_loss': train_loss, 'train_acc': train_acc,
                'val_loss': val_loss, 'val_acc': val_acc,
                'val_f1': val_f1, 'val_precision': val_precision, 'val_recall': val_recall,
                'composite_score': composite_score
            }
            
            save_checkpoint(model, optimizer, scheduler, scaler, epoch + 1, metrics, True,
                          CHECKPOINT_DIR, SERIAL_NUMBER, MODEL_NAME, SEED, 'best')
        
        # Periodic checkpoints
        if (epoch + 1) % SAVE_EVERY_N_EPOCHS == 0:
            metrics = {
                'train_loss': train_loss, 'train_acc': train_acc,
                'val_loss': val_loss, 'val_acc': val_acc,
                'val_f1': val_f1, 'val_precision': val_precision, 'val_recall': val_recall,
                'composite_score': composite_score
            }
            
            save_checkpoint(model, optimizer, scheduler, scaler, epoch + 1, metrics, False,
                          CHECKPOINT_DIR, SERIAL_NUMBER, MODEL_NAME, SEED, 'intermediate')
        
        # Overfitting monitoring
        if (epoch + 1) % OVERFITTING_CHECK_INTERVAL == 0:
            overfit_check = check_overfitting(train_acc, val_acc, train_loss, val_loss)
            history['overfitting_checks'].append((epoch + 1, overfit_check))
            
            if overfit_check['is_overfitting']:
                print(f"\n⚠️  OVERFITTING WARNING [{overfit_check['severity']}]:")
                print(f"   Train-Val Acc Gap: {overfit_check['acc_gap']:.2f}%")
                print(f"   Val-Train Loss Gap: {overfit_check['loss_gap']:.4f}")
        
        # Epoch summary
        epoch_time = time.time() - epoch_start
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"  Train: Loss={train_loss:.4f}, Acc={train_acc:.2f}%")
        print(f"  Val:   Loss={val_loss:.4f}, Acc={val_acc:.2f}%")
        print(f"  Val:   F1={val_f1:.2f}%, Prec={val_precision:.2f}%, Rec={val_recall:.2f}%")
        print(f"  Composite Score: {composite_score:.2f}")
        if is_best:
            print(f"  🎯 NEW BEST MODEL!")
        print(f"  LR: {optimizer.param_groups[0]['lr']:.6f} | Time: {epoch_time:.1f}s")
        print("=" * 70)

except KeyboardInterrupt:
    print("\n\n⚠️  TRAINING INTERRUPTED")
    print(f"Completed {epoch + 1}/{NUM_EPOCHS} epochs")
    if best_epoch > 0:
        print(f"Best model saved at epoch {best_epoch}")

# Save final checkpoint
final_metrics = {
    'train_loss': train_loss, 'train_acc': train_acc,
    'val_loss': val_loss, 'val_acc': val_acc,
    'val_f1': val_f1, 'composite_score': composite_score
}

save_checkpoint(model, optimizer, scheduler, scaler, epoch + 1, final_metrics, False,
              CHECKPOINT_DIR, SERIAL_NUMBER, MODEL_NAME, SEED, 'last')

# Training complete
total_time = time.time() - start_time
hours = int(total_time // 3600)
minutes = int((total_time % 3600) // 60)

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80)

if best_epoch > 0:
    print(f"Best model: Epoch {best_epoch}")
    print(f"  Composite Score: {best_composite_score:.2f}")
    print(f"  Val Accuracy: {best_val_acc:.2f}%")
    print(f"  Val Loss: {best_val_loss:.4f}")

print(f"\nTotal training time: {hours}h {minutes}m")
print(f"Serial: {SERIAL_NUMBER:02d}")
print(f"Checkpoints: {CHECKPOINT_DIR}")
print("="*80)

# Save training history
history_file = CHECKPOINT_DIR / f"{SERIAL_NUMBER:02d}_{MODEL_NAME}_seed{SEED}_history.json"
with open(history_file, 'w') as f:
    history_serializable = {k: [float(x) if isinstance(x, (np.floating, np.integer)) else x
                                for x in v] if isinstance(v, list) else v
                           for k, v in history.items()}
    json.dump(history_serializable, f, indent=2)

print(f"\nTraining history saved: {history_file.name}")
print("\nUse Master_Evaluation.ipynb for test set evaluation")
print("="*80)


STARTING TRAINING - VIT_B16
Serial: 12 | Seed: 42 | Epochs: 50
Device: cuda
Val size: 8,369 images
Effective batch size: 128

Epoch [1/50]
----------------------------------------------------------------------


💾 Saved best: 12_vit_b16_seed42_epoch1_best_20260116_031357.pth

Epoch 1 Summary:
  Train: Loss=1.0386, Acc=61.08%
  Val:   Loss=0.6148, Acc=84.25%
  Val:   F1=75.13%, Prec=74.73%, Rec=76.64%
  Composite Score: 79.30
  🎯 NEW BEST MODEL!
  LR: 0.000300 | Time: 133.4s

Epoch [2/50]
----------------------------------------------------------------------


💾 Saved best: 12_vit_b16_seed42_epoch2_best_20260116_031611.pth

Epoch 2 Summary:
  Train: Loss=0.5707, Acc=81.86%
  Val:   Loss=0.4340, Acc=85.98%
  Val:   F1=78.94%, Prec=77.53%, Rec=83.88%
  Composite Score: 87.02
  🎯 NEW BEST MODEL!
  LR: 0.000299 | Time: 133.5s

Epoch [3/50]
----------------------------------------------------------------------


💾 Saved best: 12_vit_b16_seed42_epoch3_best_20260116_031827.pth

Epoch 3 Summary:
  Train: Loss=0.4735, Acc=85.38%
  Val:   Loss=0.3975, Acc=90.33%
  Val:   F1=83.69%, Prec=83.44%, Rec=85.02%
  Composite Score: 89.78
  🎯 NEW BEST MODEL!
  LR: 0.000297 | Time: 136.3s

Epoch [4/50]
----------------------------------------------------------------------


💾 Saved best: 12_vit_b16_seed42_epoch4_best_20260116_032043.pth

Epoch 4 Summary:
  Train: Loss=0.4268, Acc=86.94%
  Val:   Loss=0.3459, Acc=91.00%
  Val:   F1=84.86%, Prec=83.57%, Rec=86.49%
  Composite Score: 90.70
  🎯 NEW BEST MODEL!
  LR: 0.000295 | Time: 135.5s

Epoch [5/50]
----------------------------------------------------------------------


...


Epoch 46 Summary:
  Train: Loss=0.0985, Acc=96.79%
  Val:   Loss=0.1511, Acc=96.46%
  Val:   F1=93.79%, Prec=92.68%, Rec=95.04%
  Composite Score: 96.63
  LR: 0.000005 | Time: 132.1s

Epoch [47/50]
----------------------------------------------------------------------



Epoch 47 Summary:
  Train: Loss=0.0957, Acc=96.86%
  Val:   Loss=0.1500, Acc=96.46%
  Val:   F1=93.77%, Prec=92.67%, Rec=95.03%
  Composite Score: 96.61
  LR: 0.000003 | Time: 132.1s

Epoch [48/50]
----------------------------------------------------------------------


💾 Saved best: 12_vit_b16_seed42_epoch48_best_20260116_045753.pth

Epoch 48 Summary:
  Train: Loss=0.0924, Acc=96.98%
  Val:   Loss=0.1521, Acc=96.61%
  Val:   F1=94.07%, Prec=93.05%, Rec=95.20%
  Composite Score: 96.75
  🎯 NEW BEST MODEL!
  LR: 0.000001 | Time: 132.6s

Epoch [49/50]
----------------------------------------------------------------------



Epoch 49 Summary:
  Train: Loss=0.0917, Acc=96.88%
  Val:   Loss=0.1537, Acc=96.57%
  Val:   F1=94.00%, Prec=92.99%, Rec=95.10%
  Composite Score: 96.73
  LR: 0.000000 | Time: 132.0s

Epoch [50/50]
----------------------------------------------------------------------


💾 Saved intermediate: 12_vit_b16_seed42_epoch50_intermediate_20260116_050218.pth

Epoch 50 Summary:
  Train: Loss=0.0943, Acc=96.94%
  Val:   Loss=0.1532, Acc=96.57%
  Val:   F1=94.00%, Prec=92.99%, Rec=95.10%
  Composite Score: 96.71
  LR: 0.000000 | Time: 132.6s
💾 Saved last: 12_vit_b16_seed42_epoch50_last_20260116_050218.pth

TRAINING COMPLETE
Best model: Epoch 48
  Composite Score: 96.75
  Val Accuracy: 96.61%
  Val Loss: 0.1521

Total training time: 1h 50m
Serial: 12
Checkpoints: C:\Users\Ajant\Documents\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training\Checkpoints

Training history saved: 12_vit_b16_seed42_history.json

Use Master_Evaluation.ipynb for test set evaluation
